
<div style="text-align: center; background-color: lightblue; padding: 15px;">
    <p style="text-align: center;">بسم الله، والحمد لله، والصلاة والسلام على رسول الله وعلى آله وصحبه</p>
</div>

# THIS NOTEBOOKS AIMS TO EXPLORE THE LLMs  

In [1]:
from me_r_sy.ingestion.grib import get_grib , get_var
from me_r_sy.analysis.statistics import Statistics
from me_r_sy.llm.generator import LLM , ReportGenerator
import geopandas as gpd
import numpy as np
import yaml
import calendar
import json

/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# read the config file
config_file = "/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/config/config_2018_01.yml"
with open(config_file , "r") as f :
    config = yaml.safe_load(f)

In [3]:
year       = config["date"]["year"]
month      = config["date"]["month"]
input_dir  = config["data"]["input"] 
ref_dir    = f"{input_dir}/reference_1991_2020.grib"
output_dir = config["data"]["output"] 
shp_path   = config["data"]["shapefile"]
year       = config["date"]["year"]
month      = config["date"]["month"]

In [4]:
# read the shapefile
grib_path = f"{input_dir}/input_{year}_{month:02d}.grib"

ref_path  = f"{input_dir}/reference_1991_2020.grib"

cmap_dict = {"2t": "RdYlBu_r", "tp": "GnBu", "msl": "viridis", "w": "YlOrRd"}

cmap_dict_anom = {"2t": "bwr", "tp": "PuOr", "msl": "BrBG", "w": "RdGy"}

shp_path = "/home/muhammed/METEO-REPORTING-SYSTEM/ME_R_SY/data/regions/regions.shp"

shp = gpd.read_file(shp_path)
ds        = get_grib(grib_path)
ds_ref    = get_grib(ref_path)
ref_mean  = list(map(lambda ds : Statistics.ref_avg(ds , month) , ds_ref))
month_range = calendar.monthrange(year , month)[1]

/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/home/muhammed/anaconda3/envs/me_r_sy/lib/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **

In [5]:
variables = get_var(ds)
ref_variables = get_var(ref_mean)
ref_variables["tp"] = ref_variables["tp"] * month_range

In [6]:
Stats = Statistics()
# Plot = Plotter(shp , cmap_dict , cmap_dict_anom)
statistics  = Stats.calculate_statistics(variables = variables)
daily_stats = Stats.calculate_daily(variables)
anomaly     = Stats.calculate_anomaly(statistics , ref_variables)

In [7]:
llm = LLM(daily_stats , anomaly , year , month )
generator = ReportGenerator()
message = llm.create_message()
# json_message = llm.to_json()

In [9]:
print(message)

{
    "studied_period": {
        "year": 2018,
        "month": 1
    },
    "reference_period": {
        "start_year": 1991,
        "end_year": 2020
    },
    "max": {
        "t2m": {
            "lon": -3.25,
            "lat": 22.0,
            "values": 33.631988525390625,
            "day": "2018-01-30T15:00:00.000000000"
        },
        "u10": {
            "lon": -3.0,
            "lat": 35.75,
            "values": 15.927001953125,
            "day": "2018-01-06T18:00:00.000000000"
        },
        "v10": {
            "lon": -7.0,
            "lat": 34.5,
            "values": 13.406509399414062,
            "day": "2018-01-06T03:00:00.000000000"
        },
        "msl": {
            "lon": -18.0,
            "lat": 37.0,
            "values": 1039.9837646484375,
            "day": "2018-01-26T23:00:00.000000000"
        },
        "w": {
            "lon": -10.5,
            "lat": 30.5,
            "values": 19.361127853393555,
            "day": "2018-01-26T21:0

In [11]:
sections = report
sections["summary"]
sections["temperature"]
sections["precipitation"]
sections["pressure"]
sections["wind"]

"Les valeurs extrêmes du vent enregistrées durant le mois de janvier 2018 s'élèvent à 19,4 m/s le 26 janvier à 21h au point géographique 30,5°N, -10,5°O, tandis qu'une vitesse minimale de 0,4 m/s est observée le 18 janvier à 15h au point 29,75°N, -5,0°O. Une anomalie positive de 2,58 m/s par rapport à la moyenne saisonnière est détectée, indiquant une intensité du vent légèrement supérieure"

In [ ]:
tex_doc = f"""
\section{Synthèse générale}

{{summary}}

\section{Températures}

{{sections["temperature"]}}

\begin{figure}[H]
    \centering
    \includegraphics[width=\textwidth]
    {temperature.png}
    \caption{Températures}
\end{figure}

\section{Précipitations}

{{precipitation}}

\begin{figure}[H]
    \centering
    \includegraphics[width=\textwidth]
    {figures/precipitation.png}
    \caption{Précipitations}
\end{figure}

"""